# RAG with Qdrant (Chapter 8)

This notebook accompanies Chapter 8 §8.3.1 of *Context Engineering with DSPy*. It swaps the in-memory `dspy.retrievers.Embeddings` for a Qdrant-backed retriever via `dspy-qdrant`. The `SimpleRAG` module from the previous notebook is reused unchanged — that is the point of DSPy's retriever abstraction.

**Required environment variable**

- `OPENAI_API_KEY` — used for the LM and the embedder.

**Service dependency: a Qdrant server**

Run Qdrant locally before executing the cells below:

```bash
docker run --rm -p 127.0.0.1:6333:6333 qdrant/qdrant
```

We bind to the loopback interface (`127.0.0.1`) rather than `0.0.0.0` so you do not expose Qdrant on your LAN. If the server is not reachable, the connection cell will print a clear error and the rest of the notebook will short-circuit instead of crashing.

In [ ]:
%pip install -r ../requirements.txt -q

In [ ]:
%pip install dspy-qdrant qdrant-client -q

In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM("openai/gpt-5-mini"))

## Connecting to Qdrant and seeding a collection

We point `QdrantClient` at the local container, create a tiny collection, and upsert a handful of embedded passages. The collection is small so the cell runs in seconds.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.http import models as qmodels

QDRANT_URL = "http://127.0.0.1:6333"
COLLECTION = "knowledge_base"

corpus = [
    "DSPy is a framework for programming language models.",
    "Signatures define input/output specifications for DSPy tasks.",
    "Optimizers like BootstrapFewShot automatically improve prompts.",
    "ReAct agents use tools to gather information iteratively.",
    "GEPA is a reflective evolutionary optimizer for DSPy programs.",
    "dspy.History stores conversation turns as a list of message dicts.",
]

qdrant_ready = False
client = None

try:
    client = QdrantClient(url=QDRANT_URL)
    embedder = dspy.Embedder("openai/text-embedding-3-small", dimensions=512)
    vectors = embedder(corpus)

    client.recreate_collection(
        collection_name=COLLECTION,
        vectors_config=qmodels.VectorParams(size=512, distance=qmodels.Distance.COSINE),
    )
    client.upsert(
        collection_name=COLLECTION,
        points=[
            qmodels.PointStruct(
                id=i,
                vector=list(map(float, vectors[i])),
                payload={"document": text},
            )
            for i, text in enumerate(corpus)
        ],
    )
    qdrant_ready = True
    print(f"Seeded {len(corpus)} documents into Qdrant collection '{COLLECTION}'.")
except Exception as exc:
    print(
        "Could not reach Qdrant at "
        f"{QDRANT_URL}. Start it with:\n"
        "  docker run --rm -p 127.0.0.1:6333:6333 qdrant/qdrant\n"
        f"Underlying error: {exc!r}"
    )

## Using `QdrantRM` as a drop-in retriever

The `SimpleRAG` module is identical to the in-memory notebook — only the retriever instance changes.

In [ ]:
from dspy_qdrant import QdrantRM


class SimpleRAG(dspy.Module):
    def __init__(self, retriever):
        super().__init__()
        self.retriever = retriever
        self.respond = dspy.ChainOfThought("context, question -> response")

    def forward(self, question):
        context = self.retriever(question).passages
        return self.respond(context=context, question=question)


if qdrant_ready:
    retriever = QdrantRM(
        COLLECTION,
        qdrant_client=client,
        k=5,
    )

    # Same constructor signature as the in-memory retriever.
    rag = SimpleRAG(retriever=retriever)
    result = rag(question="How do I optimize a DSPy program?")
    print(result.response)
else:
    print("Skipping retrieval — Qdrant is not running. See the setup instructions above.")